<a href="https://colab.research.google.com/github/hiiamlars/einfuehrung_in_die_programmierung_mit_MATLAB/blob/main/tvd_mi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [2]:
# @title Dependencies

# Installations of third-party libraries
!pip install openai -q
!pip install anthropic -q
!pip install together -q

# First-party imports
import os
import json
import csv

# Third-party imports
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from together import Together
from google.colab import drive

In [3]:
# @title Paths

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Thesis/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create PAPER_DIR for paper content
PAPER_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(PAPER_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {PAPER_DIR}.")

# Define and create LLM_DIR for llm based reviews
LLM_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(LLM_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {LLM_DIR}.")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Thesis/.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/iclr/final.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/llm_reviews.


In [4]:
# @title Definitions

# --- Setup ---
MODEL = "gpt-4o-mini"
#MODEL = "claude-3-5-haiku-20241022"
#MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

TIMEOUT_SECONDS = 1200

# openai_client = OpenAI(api_key=OPENAI_API_KEY)
# anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
# together_client = Together(api_key=TOGETHER_API_KEY)

In [8]:
# @title Load data

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(PAPER_DIR, "dataset_paper.parquet"))

# Select 'paper_id' and the target column 'parsed_text'
paper_content = dataset_paper[['paper_id', 'parsed_text']]

# Check paper_content
print(paper_content.head(3))

# Read llm reviews
dataset_llm = pd.read_csv(os.path.join(LLM_DIR, "llm_sim_results.csv"))

# Select 'paper_id' and 'review_text'
llm_reviews = dataset_llm[['document_id', 'review_text']] # change later to paper_id

# Check dataset_llm
print(llm_reviews.head(3))


      paper_id                                        parsed_text
0  vuD2xEtxZcj  INTRODUCTION Pruning Deep Neural Networks (DNN...
1  WoByU5W5te0  INTRODUCTION Recently, representing a 3D scene...
2  XZRmNjUMj0c  INTRODUCTION Neural Architecture Search (NAS) ...
   document_id                                        review_text
0  vuD2xEtxZcj  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...
1  WoByU5W5te0  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...
2  XZRmNjUMj0c  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...


In [9]:
# @title Merge data

merged_df = pd.merge(paper_content, llm_reviews, left_on='paper_id', right_on='document_id', how='outer') # change later to paper_id
print(merged_df.head(3))
print(merged_df.shape)

     paper_id                                        parsed_text document_id  \
0  -0tPmzgXS5  INTRODUCTION Video recognition methods has evo...  -0tPmzgXS5   
1  -0tPmzgXS5  INTRODUCTION Video recognition methods has evo...  -0tPmzgXS5   
2  -0tPmzgXS5  INTRODUCTION Video recognition methods has evo...  -0tPmzgXS5   

                                         review_text  
0  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...  
1  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...  
2  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...  
(111664, 4)


In [ ]:
# @title Define functions

# Cache for critic scores to avoid redundant API calls
cache = {}

def generate_tvd_mi_prompt(task_description: str, response_a: str, response_b: str) -> str:

    """Generate prompt for TVD-MI critic"""

    prompt = f"""You are evaluating whether two responses could be from agents working on the same task.

Task description: {task_description}

Response A: {response_a}

Response B: {response_b}

Do these responses show evidence of coming from the same task/source? Consider:
- Shared specific details, facts, or entities
- Similar topics or themes
- Overlapping information that would be unlikely if from different sources

Respond in two sentences or less. Put the information gain at the end in brackets in this format:

Rate the information gain:
- [[Significant Gain]]: Clear evidence they're from the same source (many shared specifics)
- [[Little Gain]]: Some shared elements but also differences
- [[No Gain]]: No evidence of shared source (could be from completely different tasks)"""

    return prompt

def interpret_tvd_mi_response(response: str) -> float:

    """Convert LLM response to numeric score"""

    response = response.strip().lower()

    if "[significant gain]" in response:
        return 1.0
    elif "[little gain]" in response:
        return 0.25
    elif "[no gain]" in response:
        return 0.0
    else:
        # Default to no gain if response is unclear
        print(f"Warning: Unclear response '{response}', defaulting to [[No Gain]]")
        return 0.0

def get_critic_score_with_logging(text_a: str, text_b: str, task_description: str, call_info: dict, model: str = None) -> tuple:

    """Get TVD-MI critic score via API and return both score and full interaction"""

    # Create cache key from inputs
    cache_key = (text_a, text_b, task_description)

    if model is None:
        model = MODEL

    # Check cache first
    if cache_key in cache:
        # Even for cached results, return the interaction log
        score = cache[cache_key]
        interaction = {
            "cached": True,
            "text_a": text_a,
            "text_b": text_b,
            "task_description": task_description,
            "prompt": generate_tvd_mi_prompt(task_description, text_a, text_b),
            "response": f"[CACHED] Score: {score}",
            "score": score
        }
        return score, interaction

    prompt = generate_tvd_mi_prompt(task_description, text_a, text_b)

    try:
        # Route to appropriate client based on model
        if MODEL.startswith("claude"):
            # Anthropic client returns a different structure
            response = anthropic_client.messages.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=500
            )
            response_text = response.content[0].text  # Anthropic format
        elif MODEL.startswith("gpt"):  # OpenAI models
            response = openai_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=500
            )
            response_text = response.choices[0].message.content
        else:
            response = together_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=500
            )
            response_text = response.choices[0].message.content
        score = interpret_tvd_mi_response(response_text)
        cache[cache_key] = score

        # Create interaction log
        interaction = {
            "cached": False,
            "text_a": text_a,
            "text_b": text_b,
            "task_description": task_description,
            "prompt": prompt,
            "response": response_text,
            "score": score,
            "model": MODEL,
            "temperature": 0.0,
            "max_tokens": 150
        }

        return score, interaction

    except Exception as e:
        print(f"OpenAI API error: {e}")
        raise Exception(f"API call failed: {str(e)}")

In [ ]:
# @title Run TVD-MI calculation

In [ ]:
# @title Check result